In [1]:
!pip install scikit-learn nltk numpy

In [6]:
import re
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# ✅ FIXED NLTK DOWNLOAD
try:
    nltk.data.find('tokenizers/punkt')
    nltk.data.find('corpora/stopwords')
except LookupError:
    nltk.download('punkt')
    nltk.download('stopwords')


class FAQChatbot:
    def __init__(self):
        self.faq_data = [
            {"q": "What is CodeAlpha?", "a": "CodeAlpha is a leading software development company dedicated to driving innovation and excellence across emerging technologies."},
            {"q": "How can I submit my internship tasks?", "a": "You must submit your tasks through the specific submission form shared in your respective WhatsApp group."},
            {"q": "What are the criteria for receiving a certificate?", "a": "You must successfully complete a minimum of 2 or 3 assigned domain tasks to be eligible for a certificate."},
            {"q": "Where do I host my project source code?", "a": "All source code must be uploaded to a public GitHub repository named strictly 'CodeAlpha_ProjectName'."},
            {"q": "How do I reach customer support?", "a": "You can contact support via email at services@codealpha.tech or through WhatsApp at +91 9336576683."}
        ]

        self.questions = [item["q"] for item in self.faq_data]
        self.answers = [item["a"] for item in self.faq_data]

        self.stop_words = set(stopwords.words('english'))

        # ✅ Better vectorizer config
        self.vectorizer = TfidfVectorizer(
            tokenizer=self.preprocess_pipeline,
            lowercase=True
        )

        self.tfidf_matrix = self.vectorizer.fit_transform(self.questions)

    def preprocess_pipeline(self, text):
        text = text.lower()
        text = re.sub(r'[^\w\s]', '', text)
        tokens = word_tokenize(text)

        cleaned_tokens = [
            word for word in tokens
            if word not in self.stop_words and len(word) > 1
        ]

        return cleaned_tokens

    def get_response(self, user_query, similarity_threshold=0.35):
        try:
            query_vector = self.vectorizer.transform([user_query])
            similarities = cosine_similarity(query_vector, self.tfidf_matrix).flatten()

            best_match_idx = similarities.argmax()
            highest_similarity = similarities[best_match_idx]

            if highest_similarity >= similarity_threshold:
                return self.answers[best_match_idx]
            else:
                return "❌ No strong match found. Try rephrasing your question."

        except Exception as e:
            print("Error:", e)
            return "⚠️ Something went wrong. Please try again."


# ✅ Run chatbot
if __name__ == "__main__":
    bot = FAQChatbot()

    print("🤖 Chatbot engine ready.")
    print("====================================================")
    print("🤖 Enter your question below (Type 'exit' to quit)")
    print("====================================================")

    while True:
        user_input = input("\nYou: ").strip()

        if user_input.lower() in ['exit', 'quit', 'bye']:
            print("Chatbot: Goodbye!")
            break

        if not user_input:
            continue

        response = bot.get_response(user_input)
        print(f"Chatbot: {response}")

🤖 Chatbot engine ready.
🤖 Enter your question below (Type 'exit' to quit)



You:  exit


Chatbot: Goodbye!
